# Inference Pipeline

In [1]:
# install the package
# !pip install --upgrade setuptools packaging
# !pip install -e .

In [2]:
# Install TA-Lib using conda
# !conda install -c conda-forge ta-lib -y

In [ ]:
!nvidia-smi

In [4]:
import os
import time
from datetime import datetime, timedelta

import pandas as pd
import numpy as np
import torch

import warnings

from blockhouse_ml.utils.macro_model import MetaLearner
from blockhouse_ml.utils.data_handler import DataProcessor, InferenceDataHandler
from blockhouse_ml.utils.macro_model_utils import MacroTraderModel
from blockhouse_ml.utils import fetch_merge_data
from blockhouse_ml.utils.fetch_merge_data import PolygonClient
from blockhouse_ml.utils.env import TradingEnvironment, CustomTradingEnvironment, TradingEnvironmentMicro
from blockhouse_ml.utils.micro_model_utils import MicroTraderModel

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS'] = '1'
'''
Because of this error:
 [error] Disposing session as kernel process died ExitCode: 3, Reason: OMP: Error #15: Initializing libiomp5md.dll, but found libomp140.x86_64.dll already initialized.
OMP: Hint This means that multiple copies of the OpenMP runtime have been linked into the program. That is dangerous, since it can degrade performance or cause incorrect results. The best thing to do is to ensure that only a single OpenMP runtime is linked into the process, e.g. by avoiding static linking of the OpenMP runtime in any library. As an unsafe, unsupported, undocumented workaround you can set the environment variable KMP_DUPLICATE_LIB_OK=TRUE to allow the program to continue to execute, but that may cause crashes or silently produce incorrect results. For more information, please see http://www.intel.com/software/products/support/.
'''

In [6]:
warnings.filterwarnings("ignore")

In [7]:
## Initialize the variables
# Initialize the model directory, where the models will be saved
MACRO_MODEL_DIR = 'Models'
os.makedirs(MACRO_MODEL_DIR, exist_ok=True)

MICRO_MODEL_DIR = 'MicroModels'
os.makedirs(MICRO_MODEL_DIR, exist_ok=True)


# Initialize the data directory, where the data will be stored
data_dir = 'TempData'
os.makedirs(data_dir, exist_ok=True)

# Initialize the MacroTraderModel
macro_trader = MacroTraderModel(MACRO_MODEL_DIR)

# Initialize the MicroTraderModel
micro_trader = MicroTraderModel(MICRO_MODEL_DIR)

# Initialize the data client
data_client = PolygonClient(data_dir)

# Initialize the data processor
data_processor = DataProcessor()

# Inference data handler
inference_data_handler = InferenceDataHandler()

# Initialize the MetaLearner
meta = MetaLearner()


In [8]:
# For getting the data in real time 
ticker = 'AAPL'

# Get today's date
end_date = datetime.today().strftime('%Y-%m-%d')
# Calculate the start date (7 days before today's date)
start_date = (datetime.today() - timedelta(days=2)).strftime('%Y-%m-%d')

timeframe=500
inventory=10000


In [9]:
def get_schedule(timeframe, transaction_size, market_cap_int, input_row, data):
    """
    Generates a trading schedule based on the transaction size and input data.

    Args:
    - timeframe (int): The timeframe for trade execution.
    - transaction_size (int): The size of the transaction.
    - input_row (pd.DataFrame): The input data row with forecasts and technical indicators.

    Returns:
    - list: A list of trades executed based on the generated schedule.
    """
    print("LOGGING: Generating Schedule...")
    trades, micro_input = macro_trader.infer_macro(timeframe, transaction_size, market_cap_int,  input_row, data, meta=meta, inference_data_handler=inference_data_handler)
    return trades, micro_input

# Start inferencing

In [10]:
def run_pipeline(ticker, start_timestamp, end_timestamp, timeframe, inventory, trade_set_counter=1):
    """
    Runs the entire pipeline for the trading model, including data retrieval, 
    technical indicator addition, forecasting, and generating trade schedules.

    Returns:
    - list: The list of trades generated by the model.
    """
    start_time = time.time()
    # Create the trading environment
    data = data_client.fetch_and_merge_data(ticker,start_date=start_timestamp,end_date=end_timestamp)
    data = data_processor.add_technical_indicators(data)
    input_row = inference_data_handler.add_forecasts(data)
    market_cap_int = data_client.get_market_cap(ticker)

    print(input_row)
    
    trades, micro_input = get_schedule(timeframe, inventory, market_cap_int, input_row, data)
    micro_input['Timestamp'] = trades['timestamp'].tolist()
    # Display the column names of the trades DataFrame
    # print(micro_input)
    # macro_timestamps = trades['timestamp'].tolist()
    micro_input['Ticker'] = [ticker]*len(micro_input)
    micro_input['Inventory'] = [inventory]*len(micro_input)
    micro_input['Trade_Set_ID'] = [f"set_{trade_set_counter}"]*len(micro_input)


    # ## run micro trader
    micro_env = TradingEnvironmentMicro(micro_input, preferred_timeframe=timeframe, initial_inventory=inventory)
    micro_model = micro_trader.load_model(micro_env, {}) 
 
    # Reset the environment to start inference
    obs = micro_env.reset()

    # Initialize an empty list to store trade details
    trade_details = []

    for _ in range(len(micro_input)):
        # Predict the action to take based on the current observation
        action, _states = micro_model.predict(obs)
        
        # Step through the micro_environment using the predicted action
        obs, rewards, done, info = micro_env.step(action)
        
        # Extract and print the trade details
        order_type = "Market" if action[0] < 0.5 else "Limit"
        execution_price = action[1] if order_type == "Limit" else obs[3]  # Assuming 'close' price is at index 3
        
        # Append the trade details to the list
        trade_details.append({
            'Order Type': order_type,
            'Limit Price': execution_price
        })
        
        # print(f"Trade Executed: {order_type} Order at Price: {execution_price}")
        
        # If the micro_environment is done, break the loop
        if done:
            break

    # Render the final state of the micro_environment
    micro_env.render()
    
    return trade_details

In [11]:
import warnings

# Suppress the specific ConvergenceWarning
warnings.filterwarnings("ignore", message="The optimizer returned code 4. The message is:\nInequality constraints incompatible")

# Your code that generates the warning


In [ ]:
# import warnings

# # Ignore all warnings
# warnings.filterwarnings("ignore")

import warnings

# Suppress the specific ConvergenceWarning
warnings.filterwarnings("ignore", message="The optimizer returned code 4. The message is:\nInequality constraints incompatible")

# Your code that might generate warnings

ticker = 'AAPL'

# Get today's date
end_date = (datetime.today() - timedelta(hours=2)).strftime('%Y-%m-%d')
# Calculate the start date (7 days before today's date)
start_date = (datetime.today() - timedelta(days=1)).strftime('%Y-%m-%d')

print(f"start_date: {start_date}, end_date: {end_date}")
timeframe=90 # 390
inventory=10000
trade_details = run_pipeline(ticker, start_date, end_date, timeframe, inventory)

In [ ]:
d = d